In [1]:
import os
from shutil import copyfile
from xia2pipe.connector import SQL
import numpy as np
from tqdm import tqdm

import pandas

In [10]:
def compute_volume(df):
    # only for monoclinic...
    df['volume'] = df['a'] * df['b'] * df['c'] * np.sin(np.radians(df['beta'])) / 1000. # nm^3
    return

In [2]:
password = os.environ["SARSPASS"]

db = SQL({
          "host":               "cfeld-vm05.desy.de",
          "database":           "SARS_COV_2_Analysis_v2",
          "user":               "reader",
          "password":           password,
          "connection_timeout": 60,
          "auth_plugin":        "mysql_native_password",
          'autocommit':         True, 
})

upsteam_db = SQL({
          "host":               "cfeld-vm05.desy.de",
          "database":           "SARS_COV_2_v2",
          "user":               "reader",
          "password":           password,
          "connection_timeout": 60,
          "auth_plugin":        "mysql_native_password",
          'autocommit':         True, 
})

In [3]:
TARGET = 'MPro'
METHOD = 'dmpl2-aligned'
#METHOD = 'dmpl2-qfit'

MIN_RES   = 2.2
MIN_RFREE = 0.25

FILTER_OUT_HITS = True

In [4]:
qry = db.select(['refinement_id, data_reduction_id, final_pdb_path, refinement_mtz_path, rfree, rwork, resolution_cut'],
                'Refinement',
                {'method' : METHOD})

In [9]:
# update with info from Data_Reduction table
for q in tqdm(qry):
    
    red = db.select(['crystal_id, run_id, a, b, c, alpha, beta, gamma, method, mtz_path'], 
                    'Data_Reduction', 
                    {'data_reduction_id' : q['data_reduction_id']})
    q.update(red[0])
    
    red = upsteam_db.select(['beamline'], 
                             'Diffractions', 
                            {'crystal_id' : q['crystal_id']})
    q.update(red[0])
    
    red = upsteam_db.select(['target_id, compound_drop_id', 'crystal_id', 'creator_id'], 
                             'Crystals', 
                            {'crystal_id' : q['crystal_id']})
    q.update(red[0])
    
    red = upsteam_db.select(['compound_id'], 
                             'Compound_Drops', 
                            {'compound_drop_id' : q['compound_drop_id']})
    q.update(red[0])
    
    red = upsteam_db.select(['name'], 
                         'Compounds', 
                        {'compound_id' : q['compound_id']})
    q.update(red[0])

100%|██████████| 14355/14355 [01:15<00:00, 190.91it/s]


In [11]:
# make sure we have a unique set of crystal_id/run_id
# by choosing the best rfree refinement result for each xtal

filtered_qry = {}

for q in qry:
    
    if q['target_id'].lower() == TARGET.lower():
    
        k = (q['crystal_id'], q['run_id'])

        if k in filtered_qry.keys():
            if q['rfree'] < filtered_qry[k]['rfree']:
                filtered_qry[k] = q
        else:
            filtered_qry[k] = q

print(len(qry), len(filtered_qry))

14355 5150


In [19]:
prior_to_quality_selection = pandas.DataFrame(filtered_qry.values())
compute_volume(prior_to_quality_selection)

In [23]:
prior_to_quality_selection.to_csv("./unique_mpro_datasets_prior_to_downselection.csv", index=False)

In [24]:
with open('hits.txt', 'r') as f:
    hits = f.read().split('\n')

def is_hit(qry):
    return qry['compound_id'] in hits

In [25]:
# format as string for transfer to other python env

panda_res = []

for q in filtered_qry.values():
    if q['rfree'] and q['resolution_cut']:
        
        if (q['rfree'] < MIN_RFREE) and \
           (q['resolution_cut'] < MIN_RES) and \
           q['final_pdb_path'].endswith('_003_aligned.pdb') and \
           (not FILTER_OUT_HITS or not is_hit(q)):
            panda_res.append(q)   
            
print(len(panda_res))

1147


In [26]:
df = pandas.DataFrame(panda_res)
compute_volume(df)

df['dataset_name'] = df['final_pdb_path'].apply(os.path.basename)
df['dataset_name'] = df['dataset_name'].apply(lambda x : x[:-4])

df = df[[
    'dataset_name',
    'refinement_id', 
    'data_reduction_id',
    'crystal_id',
    'run_id',
    'beamline',
    'creator_id',
    'final_pdb_path',
    'mtz_path',
    'refinement_mtz_path',
    'compound_drop_id', 'compound_id', 'name', 
    'method',
    'rfree',
    'rwork',
    'resolution_cut',
    'a',
    'b',
    'c',
    'alpha',
    'beta',
    'gamma',        
    'volume'
]]

df

,dataset_name,refinement_id,data_reduction_id,crystal_id,run_id,beamline,creator_id,final_pdb_path,mtz_path,refinement_mtz_path,...,rfree,rwork,resolution_cut,a,b,c,alpha,beta,gamma,volume
0,l7p01_16_001_003_aligned,57379,9772,MPro_1000_0,1,p11,Susanne/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2281,0.1921,1.750196,113.705011,53.312778,44.646043,90.0,103.075237,90.0,263.624500
1,l6p23_05_001_003_aligned,57383,9776,MPro_1005_0,1,p11,Ilona/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2179,0.1895,1.469575,113.385946,53.345109,44.662966,90.0,103.007571,90.0,263.215873
2,l6p23_10_001_003_aligned,57385,9780,MPro_1010_0,1,p11,Ilona/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2108,0.1817,1.420112,113.030331,53.120003,44.642397,90.0,102.981136,90.0,261.190584
3,l6p23_13_001_003_aligned,57387,9782,MPro_1013_0,1,p11,Ilona/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2020,0.1800,1.379733,112.828683,53.031398,44.730019,90.0,102.876152,90.0,260.910329
4,l6p22_16_001_003_aligned,65288,21479,MPro_1024_1,1,p11,Najeeb/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/proc...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2457,0.2069,1.750000,113.270000,53.270000,44.490000,90.0,102.956000,90.0,261.613890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1142,l2p17_01_001_003_aligned,65495,20301,MPro_132_0,1,p11,Susanne/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/proc...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2360,0.2059,1.520000,112.970000,53.040000,44.590000,90.0,102.904000,90.0,260.432634
1143,l2p22_16_001_003_aligned,65764,20384,MPro_167_1,1,p11,Najeeb/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/proc...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2443,0.2120,1.800000,113.740000,53.540000,44.560000,90.0,103.018000,90.0,264.380357
1144,l11p20_15_002_003_aligned,66037,19803,MPro_2103_0,2,None,Ilona/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/proc...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2373,0.2132,1.610000,112.760000,53.100000,44.630000,90.0,102.715000,90.0,260.671463
1145,l11p22_10_001_003_aligned,66134,19826,MPro_2232_0,1,p11,Nadine/Christina,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/proc...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2445,0.2224,1.740000,113.700000,53.570000,44.630000,90.0,102.777000,90.0,265.106092


In [27]:
csv_file = 'pdbs_{}_{}.csv'.format(TARGET, METHOD)
print('-->', csv_file)
df.to_csv(csv_file, index=False)

--> pdbs_MPro_dmpl2-aligned.csv


In [28]:
pandas.set_option('display.max_colwidth', 1000)
pandas.set_option('display.max_rows', 5)
df

,dataset_name,refinement_id,data_reduction_id,crystal_id,run_id,beamline,creator_id,final_pdb_path,mtz_path,refinement_mtz_path,...,rfree,rwork,resolution_cut,a,b,c,alpha,beta,gamma,volume
0,l7p01_16_001_003_aligned,57379,9772,MPro_1000_0,1,p11,Susanne/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/l7p01_16/l7p01_16_001/l7p01_16_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS/l7p01_16/l7p01_16_001/DataFiles/SARSCOV2_l7p01_16_001_free.mtz,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/l7p01_16/l7p01_16_001/l7p01_16_001_003_aligned.mtz,...,0.2281,0.1921,1.750196,113.705011,53.312778,44.646043,90.0,103.075237,90.0,263.624500
1,l6p23_05_001_003_aligned,57383,9776,MPro_1005_0,1,p11,Ilona/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/l6p23_05/l6p23_05_001/l6p23_05_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS/l6p23_05/l6p23_05_001/DataFiles/SARSCOV2_l6p23_05_001_free.mtz,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/l6p23_05/l6p23_05_001/l6p23_05_001_003_aligned.mtz,...,0.2179,0.1895,1.469575,113.385946,53.345109,44.662966,90.0,103.007571,90.0,263.215873
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,l11p22_10_001_003_aligned,66134,19826,MPro_2232_0,1,p11,Nadine/Christina,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/XDSv1_noice_dmpl2/l11p22_10/l11p22_10_001/l11p22_10_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/processed/reindex1_noice/l11p22_10/l11p22_10_001/full/l11p22_10_001_F.mtz,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/XDSv1_noice_dmpl2/l11p22_10/l11p22_10_001/l11p22_10_001_003_aligned.mtz,...,0.2445,0.2224,1.740000,113.700000,53.570000,44.630000,90.0,102.777000,90.0,265.106092
1146,l2p06_13_003_003_aligned,67516,20178,MPro_54_0,3,None,Susanne/Arwen,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/XDSv1_noice_dmpl2/l2p06_13/l2p06_13_003/l2p06_13_003_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/processed/reindex1_noice/l2p06_13/l2p00_16_003/full/l2p00_16_003_F.mtz,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/XDSv1_noice_dmpl2/l2p06_13/l2p06_13_003/l2p06_13_003_003_aligned.mtz,...,0.2380,0.2111,1.450000,112.350000,52.740000,44.760000,90.0,102.868000,90.0,258.557412


In [29]:
# fetch the pdbs & mtzs for tarballing and archiving

os.mkdir('./selected_dataset_archive_2024-08-07')
os.mkdir('./selected_dataset_archive_2024-08-07/pdb')
os.mkdir('./selected_dataset_archive_2024-08-07/map')
os.mkdir('./selected_dataset_archive_2024-08-07/mtz')
df.to_csv('./selected_dataset_archive_2024-08-07/metadata.csv', index=False)

for ir,r in tqdm(df.iterrows()):
    try:
        
        if r['mtz_path'].startswith('/asap3/petra3/gpfs/p11/2020/data/11010091'):
            mtz_path = r['mtz_path'].replace('11010091', '11009999')
        else:
            mtz_path = r['mtz_path']
        
        dataset_name = os.path.basename(r['final_pdb_path'])[:-4]
        
        copyfile(r['final_pdb_path'],      os.path.join('./selected_dataset_archive_2024-08-07/pdb', dataset_name + '.pdb'))
        copyfile(mtz_path,                 os.path.join('./selected_dataset_archive_2024-08-07/mtz', dataset_name + '_data.mtz'))
        copyfile(r['refinement_mtz_path'], os.path.join('./selected_dataset_archive_2024-08-07/map', dataset_name + '_map.mtz'))
    except Exception as e:
        print(e)

FileExistsError: [Errno 17] File exists: './selected_dataset_archive_2024-08-07'

## extract the hits, look for one that has a lot of datasets

In [30]:
# format as string for transfer to other python env
pandas.set_option('display.max_rows', 300)

panda_res_hits = []

for q in filtered_qry.values():
#    if q['rfree'] and q['resolution_cc']:
        
#         if (q['rfree'] < MIN_RFREE) and \
#            (q['resolution_cc'] < MIN_RES) and \
#            q['final_pdb_path'].endswith('_003_aligned.pdb') and \
        if is_hit(q):
            panda_res_hits.append(q)   
            
print(len(panda_res_hits))

df_hits = pandas.DataFrame(panda_res_hits)
df_hits = df_hits.sort_values('compound_id', axis=0)
df_hits

256


,refinement_id,data_reduction_id,final_pdb_path,refinement_mtz_path,rfree,rwork,resolution_cut,crystal_id,run_id,a,...,beta,gamma,method,mtz_path,target_id,compound_drop_id,creator_id,compound_id,name,beamline
73,60119,23250,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_5309_0/MPro_5309_0_001/MPro_5309_0_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_5309_0/MPro_5309_0_001/MPro_5309_0_001_003_aligned.mtz,0.2063,0.1768,1.440046,MPro_5309_0,1,113.387630,...,103.065797,90.0,DIALS-dials,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS/MPro_5309_0/MPro_5309_0_001/DataFiles/SARSCOV2_MPro_5309_0_001_free.mtz,MPro,5309,Susanne/Robin,DOM_SIM_015,None,p11
47,63243,35941,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_3926_0/MPro_3926_0_001/MPro_3926_0_001_001_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_3926_0/MPro_3926_0_001/MPro_3926_0_001_001_aligned.mtz,0.2645,0.2341,2.233000,MPro_3926_0,1,114.070000,...,101.606000,90.0,staraniso,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/autoproc1/MPro_3926_0/MPro_3926_0_001/staraniso_alldata-unique.mtz,MPro,3926,Susanne/Huijong,DOM_SIM_015,None,p11
51,63373,36121,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_4110_1/MPro_4110_1_001/MPro_4110_1_001_001_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_4110_1/MPro_4110_1_001/MPro_4110_1_001_001_aligned.mtz,0.2726,0.2294,2.379000,MPro_4110_1,1,113.326000,...,101.025000,90.0,staraniso,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/autoproc1/MPro_4110_1/MPro_4110_1_001/staraniso_alldata-unique.mtz,MPro,4110,Nadine/arwen,DOM_SIM_072,None,p11
50,59496,12447,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_4110_0/MPro_4110_0_001/MPro_4110_0_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_4110_0/MPro_4110_0_001/MPro_4110_0_001_003_aligned.mtz,0.2685,0.2263,2.128620,MPro_4110_0,1,113.899263,...,101.071360,90.0,DIALS-dials,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS/MPro_4110_0/MPro_4110_0_001/DataFiles/SARSCOV2_MPro_4110_0_001_free.mtz,MPro,4110,Nadine/arwen,DOM_SIM_072,None,p11
86,56333,24232,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_1p7A_dmpl2/MPro_5605_0/MPro_5605_0_001/MPro_5605_0_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_1p7A_dmpl2/MPro_5605_0/MPro_5605_0_001/MPro_5605_0_001_003_aligned.mtz,0.2543,0.2141,1.700057,MPro_5605_0,1,112.483125,...,102.811039,90.0,DIALS_1p7A-dials,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_1p7A/MPro_5605_0/MPro_5605_0_001/DataFiles/SARSCOV2_MPro_5605_0_001_free.mtz,MPro,5605,Susanne/Robin,DOM_SIM_072,None,p11
52,63377,36125,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_4114_1/MPro_4114_1_001/MPro_4114_1_001_001_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/staraniso_dmpl2/MPro_4114_1/MPro_4114_1_001/MPro_4114_1_001_001_aligned.mtz,0.2376,0.2154,2.318000,MPro_4114_1,1,113.875000,...,102.125000,90.0,staraniso,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/autoproc1/MPro_4114_1/MPro_4114_1_001/staraniso_alldata-unique.mtz,MPro,4114,Nadine/arwen,DOM_SIM_074,None,p11
87,60311,23777,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_5610_0/MPro_5610_0_001/MPro_5610_0_001_003_aligned.pdb,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_5610_0/MPro_5610_0_001/MPro_5610_0_001_003_aligned.mtz,0.2111,0.1844,1.758636,MPro_5610_0,1,114.314854,...,103.001552,90.0,DIALS-dials,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS/MPro_5610_0/MPro_5610_0_001/DataFiles/SARSCOV2_MPro_5610_0_001_free.mtz,MPro,5610,Susanne/Robin,DOM_SIM_074,None,p11
88,60312,23779,/asap3/petra3/gpfs/p11/2020/data/11009999/scratch_cc/DIALS_dmpl2/MPro_5610_2/MPro_5610_2_001/MPro_5610_2_001_001_aligned.pdb

In [31]:
selected_hit = 'TGI'

o_dir = 'hit_datasets/' + selected_hit
if not os.path.exists(o_dir):
    os.mkdir(o_dir)

#sel_df = df_hits[ df_hits['name'] == selected_hit ]
sel_df = df_hits[ df_hits['compound_id'] == 'SPE_A39935389' ]

for idx, l in sel_df.iterrows():
    copyfile(l['refinement_mtz_path'], o_dir+'/m%04d.mtz' % idx)
    copyfile(l['final_pdb_path'],      o_dir+'/m%04d.pdb' % idx)